# 04 · Calidad del dataset UN WPP 2024 (población por sexo y edad, Corea)
Rol en el proyecto: **solo contraste** de la población histórica (KOSIS) y de las proyecciones (KOSTAT). Variable de interés: población por sexo × grupo quinquenal, 2000-2072, variantes Median, High-fertility y Low-fertility.

In [1]:
import sys; from pathlib import Path; sys.path.insert(0, str(Path.cwd().parent))
import json
import pandas as pd
from src.transform import lectura_bronze as lb
from src.quality import perfilamiento as pf

pd.set_option("display.float_format", "{:,.1f}".format)
CLAVE = ["variante", "anio", "sexo", "edad"]
PERIODO = {"Median": range(2000, 2073), "High-fertility": range(2024, 2073), "Low-fertility": range(2024, 2073)}

todo = lb.unwpp(False)   # 17 variantes, tal como llegan a bronze
df = lb.unwpp()          # 3 variantes del proyecto
df.head(3)

,variante,estimateType,estimateMethod,anio,sexo,edad,ageStart,valor
0,Median,Model-based Estimates,Interpolation,2000,Male,0-4,0,"1,717,700.5"
1,Median,Model-based Estimates,Interpolation,2000,Female,0-4,0,"1,514,629.0"
2,Median,Model-based Estimates,Interpolation,2000,Both sexes,0-4,0,"3,232,329.5"


## 1. Volumen: bronze vs usado
El archivo trae 17 variantes de la ONU; el proyecto solo usa 3 (Median, High-fertility, Low-fertility).

In [2]:
vol = todo.groupby("variante").agg(filas=("valor", "size"), anio_min=("anio", "min"), anio_max=("anio", "max"))
vol["se_usa"] = vol.index.isin(list(PERIODO))
print(f"Bronze: {len(todo):,} filas, {todo['variante'].nunique()} variantes | Usado: {len(df):,} filas ({len(df)/len(todo):.1%})")
vol.sort_values("se_usa", ascending=False).head(5)

Bronze: 53,991 filas, 17 variantes | Usado: 10,773 filas (20.0%)


,filas,anio_min,anio_max,se_usa
variante,,,,
High-fertility,3087,2024,2072,True
Median,4599,2000,2072,True
Low-fertility,3087,2024,2072,True
Instant-replacement-fertility,3087,2024,2072,False
No fertility below age 18,3087,2024,2072,False


## 2. Completitud
Celda esperada = variante × sexo × edad × año dentro del periodo de diseño de cada variante.

In [3]:
comp = pd.concat([pf.completitud(df[df["variante"] == v], ["sexo", "edad"], anios).assign(variante=v)
                  for v, anios in PERIODO.items()])
resumen_comp = comp.groupby("variante").agg(series_sexo_edad=("completitud_pct", "size"), anios_esperados=("anios_esperados", "first"),
                             completitud_min=("completitud_pct", "min"), completitud_media=("completitud_pct", "mean"))

# Grilla completa: 3 sexos x 21 grupos de edad en cada variante-año
celdas = df.groupby(["variante", "anio"]).size()
print("Celdas por variante-año:", sorted(celdas.unique().tolist()), "| esperado:", 3 * df["edad"].nunique())
resumen_comp

Celdas por variante-año: [63] | esperado: 63


,series_sexo_edad,anios_esperados,completitud_min,completitud_media
variante,,,,
High-fertility,63,49,100.0,100.0
Low-fertility,63,49,100.0,100.0
Median,63,73,100.0,100.0


## 3. Validez y unicidad
Nulos, rango (≥ 0), duplicados sobre (variante, anio, sexo, edad) y tipo de estimación.

In [4]:
print("Nulos en valor:", df["valor"].isna().sum())
print("Valores < 0:", len(pf.fuera_de_rango(df, rmin=0)))
print("Valores no enteros (personas fraccionarias):", int((df["valor"] % 1 != 0).sum()), "de", len(df))
print("Mínimo:", df["valor"].min(), "| máximo:", f"{df['valor'].max():,.0f}")
dup = {v: pf.duplicados(g, CLAVE) for v, g in df.groupby("variante")}
print("Duplicados en la clave:", dup)
df.groupby(["variante", "estimateType"])["anio"].agg(["min", "max"])

Nulos en valor: 0
Valores < 0: 0
Valores no enteros (personas fraccionarias): 5536 de 10773
Mínimo: 13.0 | máximo: 4,517,670
Duplicados en la clave: {'High-fertility': 0, 'Low-fertility': 0, 'Median': 0}


,,min,max
variante,estimateType,,
High-fertility,Model-based Estimates,2024,2072
Low-fertility,Model-based Estimates,2024,2072
Median,Model-based Estimates,2000,2072


`estimateType` es constante y no separa estimado de proyectado. Ese dato está en `estimateMethod`, que `lb.unwpp()` no conserva; se revisa en el JSON crudo:

In [5]:
crudo = pd.DataFrame(lb.crudo_json(lb.archivo_reciente("unwpp", "poblacion_edad_sexo.json")))
crudo[crudo["variantId"].isin(lb.VARIANTES_UNWPP)].groupby(["variant", "estimateMethod"])["timeLabel"].agg(["min", "max"])

min   max
variant        estimateMethod            
High-fertility Projection      2024  2072
Low-fertility  Projection      2024  2072
Median         Interpolation   2000  2021
               Projection      2022  2072

## 4. Consistencia
Hombres + Mujeres = Ambos sexos; total por año (no hay fila de total: se suma la pirámide) y verificación de unidad.

In [6]:
p = df.pivot_table(index=["variante", "anio", "edad"], columns="sexo", values="valor")
dif_sexo = (p["Male"] + p["Female"] - p["Both sexes"]).abs()
incons_sexo = (dif_sexo > 1).groupby("variante").sum()
print("Celdas con |H + M - T| > 1 persona:", incons_sexo.to_dict(), "| dif. máx.:", dif_sexo.max())

tot = df[df["sexo"] == "Both sexes"].groupby(["variante", "anio"])["valor"].sum().unstack(0)
print(f"Población total 2020 (Median): {tot.loc[2020, 'Median']/1e6:.2f} M  -> unidad = personas (Corea ~51.8 M)")

# Coherencia entre variantes: High >= Median >= Low en la población total (2024-2072)
proy = tot.loc[2024:]
orden_ok = (proy["High-fertility"] >= proy["Median"]) & (proy["Median"] >= proy["Low-fertility"])
print("Años con orden High >= Median >= Low:", int(orden_ok.sum()), "de", len(proy))
print(f"Divergencia High vs Median en 2024: {(proy.loc[2024, 'High-fertility']/proy.loc[2024, 'Median']-1)*100:.2f} %")
(tot.loc[[2000, 2020, 2024, 2050, 2072]] / 1e6).round(2)

Celdas con |H + M - T| > 1 persona: {'High-fertility': 0, 'Low-fertility': 0, 'Median': 0} | dif. máx.: 4.656612873077393e-10
Población total 2020 (Median): 51.86 M  -> unidad = personas (Corea ~51.8 M)
Años con orden High >= Median >= Low: 49 de 49
Divergencia High vs Median en 2024: 0.08 %


variante,High-fertility,Low-fertility,Median
anio,,,
2000,NaN,NaN,46.8
2020,NaN,NaN,51.9
2024,51.8,51.7,51.7
2050,48.2,42.1,45.1
2072,39.6,28.1,33.4


## 5. Homologación contra las dimensiones del proyecto
Etiquetas que no existen en `edades.csv` / `sexo.csv` y su regla para silver.

In [7]:
raiz = Path.cwd().parent / "config" / "mappings"
dim_edad = set(pd.read_csv(raiz / "edades.csv")["cod_grupo_edad"])
edad_fuera = sorted(set(df["edad"]) - dim_edad, key=lambda e: int(e.rstrip("+").split("-")[0]))
homol = pd.DataFrame([
    {"campo": "sexo", "etiqueta_unwpp": "Both sexes / Male / Female", "destino": "T / H / M", "regla": "mapeo 1:1"},
    {"campo": "edad", "etiqueta_unwpp": ", ".join(edad_fuera), "destino": "85+", "regla": "sumar (abierto 85+)"},
    {"campo": "edad", "etiqueta_unwpp": "(no viene)", "destino": "TOTAL, 0-14, 15-64, 65+", "regla": "calcular en silver"},
    {"campo": "variante", "etiqueta_unwpp": "Median / High-fertility / Low-fertility", "destino": "medio / alto / bajo", "regla": "mapeo 1:1 a escenario"},
])
peso_85 = df[df["edad"].isin(edad_fuera) & (df["sexo"] == "Both sexes")].groupby(["variante", "anio"])["valor"].sum().unstack(0) / tot * 100
print("Peso de 85+ en el total (Median) 2000 / 2024 / 2072 (%):", peso_85["Median"].loc[[2000, 2024, 2072]].round(2).tolist())
homol

Peso de 85+ en el total (Median) 2000 / 2024 / 2072 (%): [0.37, 2.17, 14.98]


,campo,etiqueta_unwpp,destino,regla
0,sexo,Both sexes / Male / Female,T / H / M,mapeo 1:1
1,edad,"85-89, 90-94, 95-99, 100+",85+,sumar (abierto 85+)
2,edad,(no viene),"TOTAL, 0-14, 15-64, 65+",calcular en silver
3,variante,Median / High-fertility / Low-fertility,medio / alto / bajo,mapeo 1:1 a escenario


## 6. Resumen de calidad por variante

In [8]:
filas = []
for v, anios in PERIODO.items():
    g = df[df["variante"] == v]
    c = comp[comp["variante"] == v]["completitud_pct"]
    fuera = len(pf.fuera_de_rango(g, rmin=0)) + int(g["valor"].isna().sum())
    inc = int(incons_sexo.get(v, 0))
    ok = c.min() == 100 and fuera == 0 and dup[v] == 0 and inc == 0
    filas.append({"variante": v, "periodo": f"{g['anio'].min()}-{g['anio'].max()}", "filas": len(g),
                  "completitud_pct": c.mean(), "fuera_rango_o_nulo": fuera, "duplicados": dup[v],
                  "incons_H+M≠T": inc, "estado": "OK" if ok else "Observación"})
resumen = pd.DataFrame(filas).set_index("variante")
resumen

,periodo,filas,completitud_pct,fuera_rango_o_nulo,duplicados,incons_H+M≠T,estado
variante,,,,,,,
Median,2000-2072,4599,100.0,0,0,0,OK
High-fertility,2024-2072,3087,100.0,0,0,0,OK
Low-fertility,2024-2072,3087,100.0,0,0,0,OK


### Hallazgos y acciones para silver
- **Calidad estructural completa en las 3 variantes:** cobertura 100 % en su periodo de diseño (Median 2000-2072; High/Low 2024-2072), sin nulos, negativos ni duplicados en la clave, y H + M = T exacto. No hay bloqueantes.
- **Unidad = personas** (total 2020 Median ≈ 51,86 M): no multiplicar por 1.000. La mitad de los valores (5.536 de 10.773) no son enteros (fracciones de 0,5) → redondear a entero al cargar a silver.
- **`estimateType` no sirve para separar niveles de evidencia** (es siempre "Model-based Estimates"). Usar `estimateMethod`: Interpolation 2000-2021 = estimado; Projection 2022-2072 = proyección. Acción: que `lb.unwpp()` conserve `estimateMethod` y que silver lo use para separar histórico de proyección.
- **Homologación:** sexo `Both sexes/Male/Female` → `T/H/M`; agregar 85-89, 90-94, 95-99 y 100+ en `85+` (pesa 0,4 % del total en 2000 y 15,0 % en 2072, así que el agregado es crítico); calcular `TOTAL`, `0-14`, `15-64`, `65+` (la fuente no trae fila de total); variantes → `medio/alto/bajo`.
- **Solo contraste:** 2000-2021 contra la población estimada KOSIS y 2022-2072 contra KOSTAT por escenario, en `silver.conciliacion`; nunca reemplaza valores. High/Low ya difieren de Median en 2024 (+/-0,08 %) y mantienen High ≥ Median ≥ Low los 49 años, así que el contraste por escenario se hace año a año.
- Las 14 variantes restantes (intervalos 80/95 %, migración cero, etc.) se quedan en bronze y no pasan a silver.